# Лабораторная работа № 4 — ЭТАЛОННОЕ РЕШЕНИЕ

## Гомография, задача PnP и метод RANSAC

**Дисциплина:** Введение в технологии виртуальной дополненной реальности и робототехники
**Тема курса:** 5. Гомография, задача PnP и метод RANSAC — математика AR-трекинга

---

**Для преподавателя.** Не выдавать студентам до сдачи работы.

---

### Цель работы

Реализовать полный конвейер маркерного AR: оценить гомографию плоскости методом DLT,
восстановить из неё позу камеры, наложить виртуальный объект, уточнить позу методом
Гаусса — Ньютона и защитить оценку от выбросов методом RANSAC.

### Что понадобится

`numpy`, `matplotlib`. `opencv-python` — только для сверки в задании 4; если он
не установлен, сверку можно пропустить.

### Правило практикума

Алгоритм вы реализуете сами. `cv2.findHomography`, `cv2.solvePnP` допустимо
использовать **только** для сверки — и такую сверку нужно показать явно.

## Теоретическая справка

### Гомография плоскости

Если все наблюдаемые точки лежат в одной плоскости, связь между их координатами
на плоскости и на изображении задаётся матрицей $3\times3$ в однородных координатах:

$$\tilde{u} \sim H \tilde{x}, \qquad H \in \mathbb{R}^{3\times3},\ \det H \neq 0 .$$

Знак $\sim$ означает равенство с точностью до множителя: однородные координаты
определены с точностью до масштаба. Поэтому у $H$ не девять степеней свободы, а восемь,
и её обычно нормируют условием $h_{33} = 1$.

### DLT и роль SVD

Каждое соответствие $x \leftrightarrow u$ даёт два линейных уравнения на элементы $H$.
Складывая их в матрицу $A$, получаем однородную систему

$$A h = 0, \qquad h = \operatorname{vec}(H) \in \mathbb{R}^9 .$$

Тривиальное решение $h = 0$ не годится, поэтому ищут $h$ единичной длины,
минимизирующий $\|Ah\|$. Ответ даёт сингулярное разложение: **это правый сингулярный
вектор, отвечающий наименьшему сингулярному числу**.

Это та же схема, что в триангуляции (ЛР 3, задание 7) и в графовом SLAM (тема 14).
Запомните её — она встретится в курсе ещё дважды.

### Нормализация Хартли

Координаты пикселей имеют порядок сотен, а координаты маркера — сотых долей метра.
В матрице $A$ соседствуют элементы, различающиеся на пять порядков, и её
обусловленность оказывается плохой. Лекарство: привести обе системы точек
к центроиду в нуле и среднему расстоянию $\sqrt{2}$, решить задачу
в нормированных координатах, затем вернуться:

$$H = T_{dst}^{-1} \, \tilde{H} \, T_{src} .$$

Это не косметика: в задании 2 вы увидите улучшение обусловленности на четыре порядка.

### Поза из гомографии

Для плоскости $z = 0$ матрица проекции сокращается до трёх столбцов:

$$H \sim K\,[\,r_1 \mid r_2 \mid t\,] .$$

Отсюда $r_1, r_2, t$ восстанавливаются с точностью до общего множителя, который
находится из условия $\|r_1\| = 1$, а $r_3 = r_1 \times r_2$. Из-за шума
столбцы не будут строго ортогональны, поэтому результат проецируют на $SO(3)$
через SVD — ровно та ортогонализация, что была в ЛР 1 и ЛР 2.

### Задача PnP и метод Гаусса — Ньютона

Оценка из гомографии годится как начальное приближение, но не минимизирует
геометрически осмысленную величину. Правильная постановка — минимизация
**репроекционной ошибки**:

$$\min_{\omega,\,t} \sum_i \bigl\| \pi(K,\,e^{[\omega]_\times},\,t,\,X_i) - u_i \bigr\|^2 .$$

Это нелинейная задача наименьших квадратов. Ориентация параметризуется вектором
поворота из лекции 3 — именно потому, что метод Гаусса — Ньютона требует, чтобы
неизвестные лежали в линейном пространстве, а кватернион со связью единичной нормы
для этого не годится.

### RANSAC

Реальные соответствия содержат грубые ошибки. Один выброс портит метод наименьших
квадратов полностью, поскольку квадратичный штраф придаёт ему огромный вес. Схема
RANSAC: многократно взять минимальную выборку (для гомографии — четыре точки),
построить по ней модель, посчитать, сколько точек ей соответствуют, и оставить
лучшую гипотезу.

Число итераций, гарантирующее с вероятностью $p$ хотя бы одну чистую выборку
при доле выбросов $\varepsilon$ и размере выборки $s$:

$$N = \frac{\log(1-p)}{\log\bigl(1 - (1-\varepsilon)^s\bigr)} .$$

In [1]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)

K = np.array([[800.0, 0.0, 640.0], [0.0, 800.0, 360.0], [0.0, 0.0, 1.0]])

def rodrigues(w):
    """Формула Родрига из лекции 3 — понадобится для задания 4."""
    th = np.linalg.norm(w)
    if th < 1e-12:
        return np.eye(3)
    k = w / th
    Kx = np.array([[0, -k[2], k[1]], [k[2], 0, -k[0]], [-k[1], k[0], 0]])
    return np.eye(3) + np.sin(th) * Kx + (1 - np.cos(th)) * Kx @ Kx

## Задание 1. Гомография методом DLT

**Ответ на вопрос:** у гомографии восемь степеней свободы, каждое соответствие даёт два уравнения, поэтому нужно четыре точки. Три дают шесть уравнений — недостаточно. Кроме того, никакие три из четырёх точек не должны лежать на одной прямой, иначе система вырождается.

Ожидаемо: невязка $\sim 2{,}7\cdot10^{-10}$ пикселя и на опорных, и на контрольных точках.

In [2]:
def dlt_homography(src, dst):
    A = []
    for (x, y), (u, v) in zip(src, dst):
        A.append([-x, -y, -1, 0, 0, 0, u * x, u * y, u])
        A.append([0, 0, 0, -x, -y, -1, v * x, v * y, v])
    h = np.linalg.svd(np.asarray(A, float))[2][-1]
    H = h.reshape(3, 3)
    return H / H[2, 2]

def apply_H(H, pts):
    p = np.hstack([np.atleast_2d(pts), np.ones((len(np.atleast_2d(pts)), 1))])
    q = p @ H.T
    return q[:, :2] / q[:, 2:3]

# маркер: квадрат 0,2 м в плоскости z = 0
M = np.array([[-0.1, -0.1], [0.1, -0.1], [0.1, 0.1], [-0.1, 0.1]])
w_true = np.array([0.35, -0.55, 0.25])
R_true, t_true = rodrigues(w_true), np.array([0.02, -0.01, 0.9])

def project_plane(M2, R, t):
    X = np.hstack([M2, np.zeros((len(M2), 1))])
    Xc = X @ R.T + t
    uvw = Xc @ K.T
    return uvw[:, :2] / uvw[:, 2:3]

img = project_plane(M, R_true, t_true)
H = dlt_homography(M, img)
err1 = np.abs(apply_H(H, M) - img).max()
print('Задание 1: макс. невязка гомографии на 4 точках %.2e пикс' % err1)
assert err1 < 1e-8
Mtest = rng.uniform(-0.1, 0.1, (50, 2))
err1b = np.abs(apply_H(H, Mtest) - project_plane(Mtest, R_true, t_true)).max()
print('           на 50 контрольных точках маркера    %.2e пикс' % err1b)
assert err1b < 1e-8

Задание 1: макс. невязка гомографии на 4 точках 2.31e-10 пикс
           на 50 контрольных точках маркера    2.14e-10 пикс


## Задание 2. Нормализация Хартли

**Ответы на вопросы:** плохая обусловленность возникает из-за разного масштаба координат: пиксели порядка сотен, метры порядка сотых. В матрице $A$ соседствуют элементы, различающиеся на пять порядков.

При 30 точках и шуме 1 пиксель разница в точности мала (0,378 против 0,351), потому что задача сильно переопределена и запас устойчивости велик. Разница становится решающей при минимальной выборке из четырёх точек, при большом шуме и при вычислениях одинарной точности — а именно так работает RANSAC в задании 5, где гомография строится по четырём точкам тысячи раз.

In [3]:
def normalize_pts(p):
    c = p.mean(0)
    s = np.sqrt(2) / np.sqrt(((p - c) ** 2).sum(1)).mean()
    T = np.array([[s, 0, -s * c[0]], [0, s, -s * c[1]], [0, 0, 1]])
    return apply_H(T, p), T

def dlt_normalized(src, dst):
    sn, Ts = normalize_pts(src)
    dn, Td = normalize_pts(dst)
    Hn = dlt_homography(sn, dn)
    H = np.linalg.inv(Td) @ Hn @ Ts
    return H / H[2, 2]

def build_A(src, dst):
    A = []
    for (x, y), (u, v) in zip(src, dst):
        A.append([-x, -y, -1, 0, 0, 0, u * x, u * y, u])
        A.append([0, 0, 0, -x, -y, -1, v * x, v * y, v])
    return np.asarray(A, float)

condA = np.linalg.cond(build_A(M, img))
sn, _ = normalize_pts(M); dn, _ = normalize_pts(img)
condN = np.linalg.cond(build_A(sn, dn))
print('Задание 2: обусловленность A без нормализации %.3e' % condA)
print('           с нормализацией                    %.3e  (лучше в %.0f раз)'
      % (condN, condA / condN))
assert condN < condA / 100

Mg = rng.uniform(-0.1, 0.1, (30, 2))
ig = project_plane(Mg, R_true, t_true)
res_raw, res_nrm = [], []
for _ in range(300):
    n = ig + rng.normal(0, 1.0, ig.shape)
    for f, acc in ((dlt_homography, res_raw), (dlt_normalized, res_nrm)):
        Hh = f(Mg, n)
        acc.append(np.sqrt(np.mean((apply_H(Hh, Mg) - ig) ** 2)))
print('           СКО при шуме 1 пикс: без нормализации %.4f, с нормализацией %.4f пикс'
      % (np.mean(res_raw), np.mean(res_nrm)))

Задание 2: обусловленность A без нормализации 7.436e+04
           с нормализацией                    3.156e+00  (лучше в 23564 раз)
           СКО при шуме 1 пикс: без нормализации 0.3778, с нормализацией 0.3507 пикс


## Задание 3. Поза из гомографии и наложение виртуального объекта

**Ответы на вопросы:** ортогонализация нужна, потому что из-за шума и погрешностей восстановленные $r_1, r_2$ не строго ортогональны и не строго единичны — матрица съезжает с $SO(3)$, как обсуждалось в лекции 2. Неоднозначность знака возникает оттого, что однородные координаты определены с точностью до множителя, в том числе отрицательного; она разрешается требованием $t_z > 0$ — маркер обязан находиться перед камерой, а не позади неё.

In [4]:
def pose_from_homography(H, K):
    Ki = np.linalg.inv(K)
    h1, h2, h3 = (Ki @ H).T
    lam = 1.0 / np.linalg.norm(h1)
    r1, r2, t = lam * h1, lam * h2, lam * h3
    R = np.column_stack([r1, r2, np.cross(r1, r2)])
    U, _, Vt = np.linalg.svd(R)
    R = U @ np.diag([1, 1, np.linalg.det(U @ Vt)]) @ Vt
    if t[2] < 0:
        R, t = -R, -t
        R[:, 2] *= -1
    return R, t

R_est, t_est = pose_from_homography(H, K)
dR = np.degrees(np.arccos(np.clip((np.trace(R_est.T @ R_true) - 1) / 2, -1, 1)))
dt = np.linalg.norm(t_est - t_true)
print('Задание 3: ошибка ориентации %.3e град, положения %.3e м' % (dR, dt))
assert dR < 1e-4 and dt < 1e-8

cube = np.array([[x, y, z] for x in (-.06, .06) for y in (-.06, .06) for z in (0, -.12)])
edges = [(i, j) for i in range(8) for j in range(i + 1, 8)
         if np.sum(np.abs(cube[i] - cube[j]) > 1e-9) == 1]
Xc = cube @ R_est.T + t_est
uv = (Xc @ K.T)[:, :2] / (Xc @ K.T)[:, 2:3]
fig, ax = plt.subplots(figsize=(6.4, 3.6))
ax.plot(*np.vstack([img, img[0]]).T, color='#E08A2E', lw=2.2, label='маркер')
for i, j in edges:
    ax.plot(*zip(uv[i], uv[j]), color='#1F2733', lw=1.4)
ax.set_xlim(0, 1280); ax.set_ylim(720, 0); ax.set_aspect('equal'); ax.legend()
plt.tight_layout(); plt.savefig('/tmp/_ar.png', dpi=80); plt.close()
print('           виртуальный куб наложен, рёбер %d' % len(edges))

Задание 3: ошибка ориентации 1.708e-06 град, положения 7.534e-13 м
           виртуальный куб наложен, рёбер 12


## Задание 4. Уточнение позы методом Гаусса — Ньютона

**Ответы на вопросы:** невязка убывает квадратично — за 6 итераций с 45 пикселей до $3\cdot10^{-14}$; каждая итерация примерно удваивает число верных знаков. Это характерная скорость метода Ньютона вблизи решения.

Вектор поворота выбран потому, что метод Гаусса — Ньютона делает шаг в линейном пространстве. У кватерниона есть связь $\|q\| = 1$, и произвольный шаг из него выводит; у углов Эйлера есть вырождение. Вектор поворота свободен от обоих недостатков — это ровно тот случай, ради которого он и вводился в лекции 3.

При ошибке 90° метод может сойтись к постороннему минимуму или разойтись: сходимость Гаусса — Ньютона только локальная. Предложите студенту проверить это самому.

In [5]:
Xw = rng.uniform([-0.12, -0.12, -0.05], [0.12, 0.12, 0.05], (40, 3))
obs = None
def reproject(xi, X):
    R, t = rodrigues(xi[:3]), xi[3:]
    Xc = X @ R.T + t
    uvw = Xc @ K.T
    return uvw[:, :2] / uvw[:, 2:3]

obs = reproject(np.r_[w_true, t_true], Xw)

def pnp_gauss_newton(X, uv, xi0, iters=30, tol=1e-14):
    xi = xi0.astype(float).copy()
    hist = []
    for _ in range(iters):
        r = (reproject(xi, X) - uv).ravel()
        hist.append(np.sqrt(np.mean(r ** 2)))
        J = np.zeros((len(r), 6))
        for k in range(6):
            e = np.zeros(6); e[k] = 1e-7
            J[:, k] = ((reproject(xi + e, X) - reproject(xi - e, X)).ravel()) / 2e-7
        d = np.linalg.lstsq(J, -r, rcond=None)[0]
        xi = xi + d
        if np.linalg.norm(d) < tol:
            break
    hist.append(np.sqrt(np.mean((reproject(xi, X) - uv).ravel() ** 2)))
    return xi, hist

xi0 = np.r_[w_true + 0.25, t_true + [0.05, 0.05, 0.08]]
xi, hist = pnp_gauss_newton(Xw, obs, xi0)
print('Задание 4: итераций %d, невязка %.3e -> %.3e пикс'
      % (len(hist) - 1, hist[0], hist[-1]))
print('           ошибка ориентации %.2e град, положения %.2e м'
      % (np.degrees(np.arccos(np.clip(
          (np.trace(rodrigues(xi[:3]).T @ R_true) - 1) / 2, -1, 1))),
         np.linalg.norm(xi[3:] - t_true)))
assert hist[-1] < 1e-9
try:
    import cv2
    ok, rv, tv = cv2.solvePnP(Xw, obs, K, None, flags=cv2.SOLVEPNP_ITERATIVE)
    print('           сверка с cv2.solvePnP: расхождение по t %.2e м'
          % np.linalg.norm(tv.ravel() - xi[3:]))
except ImportError:
    print('           (OpenCV не установлен — сверка пропущена)')

Задание 4: итераций 6, невязка 4.515e+01 -> 5.364e-14 пикс
           ошибка ориентации 0.00e+00 град, положения 1.12e-16 м


           сверка с cv2.solvePnP: расхождение по t 7.76e-18 м


## Задание 5. RANSAC против выбросов

**Ответы на вопросы:** квадратичный штраф придаёт выбросу вес, пропорциональный квадрату его отклонения; один выброс с ошибкой 180 пикселей перевешивает сотню точек с ошибкой 1 пиксель. Порог согласия выбирают из уровня шума — обычно 2—3 стандартных отклонения (здесь 3 пикселя при шуме 0,5). Слишком большой порог впустит выбросы внутрь, слишком малый отбросит хорошие точки и оставит модель без опоры.

Ожидаемо: 70,3 пикселя без RANSAC против 0,077 с ним; найдены все 72 истинных соответствия, ложных нет.

In [6]:
def ransac_homography(src, dst, thr=3.0, iters=2000, seed=0):
    r = np.random.default_rng(seed)
    best_in, best_H = None, None
    n = len(src)
    for _ in range(iters):
        idx = r.choice(n, 4, replace=False)
        try:
            Hh = dlt_normalized(src[idx], dst[idx])
        except np.linalg.LinAlgError:
            continue
        d = np.linalg.norm(apply_H(Hh, src) - dst, axis=1)
        inl = d < thr
        if best_in is None or inl.sum() > best_in.sum():
            best_in, best_H = inl, Hh
    if best_in.sum() >= 4:
        best_H = dlt_normalized(src[best_in], dst[best_in])
    return best_H, best_in

def make_data(n=120, frac_out=0.4, noise=0.5, seed=1):
    r = np.random.default_rng(seed)
    S = r.uniform(-0.1, 0.1, (n, 2))
    Dt = project_plane(S, R_true, t_true) + r.normal(0, noise, (n, 2))
    k = int(n * frac_out)
    out = r.choice(n, k, replace=False)
    Dt[out] += r.uniform(-180, 180, (k, 2))
    mask = np.ones(n, bool); mask[out] = False
    return S, Dt, mask

S, Dt, inl_true = make_data()
H_plain = dlt_normalized(S, Dt)
H_rs, inl = ransac_homography(S, Dt)
clean = project_plane(S[inl_true], R_true, t_true)
e_plain = np.sqrt(np.mean((apply_H(H_plain, S[inl_true]) - clean) ** 2))
e_rs = np.sqrt(np.mean((apply_H(H_rs, S[inl_true]) - clean) ** 2))
print('Задание 5: при 40 %% выбросов — СКО без RANSAC %.1f пикс, с RANSAC %.3f пикс'
      % (e_plain, e_rs))
print('           найдено правильных соответствий %d из %d, ложных %d'
      % ((inl & inl_true).sum(), inl_true.sum(), (inl & ~inl_true).sum()))
assert e_rs < 1.0 and e_plain > 10

fracs = np.arange(0.0, 0.75, 0.05)
ep, er = [], []
for fo in fracs:
    S2, D2, m2 = make_data(frac_out=fo, seed=int(fo * 100) + 7)
    cl = project_plane(S2[m2], R_true, t_true)
    ep.append(np.sqrt(np.mean((apply_H(dlt_normalized(S2, D2), S2[m2]) - cl) ** 2)))
    Hr, _ = ransac_homography(S2, D2, seed=int(fo * 100))
    er.append(np.sqrt(np.mean((apply_H(Hr, S2[m2]) - cl) ** 2)))
print('           доля выбросов 0 %% -> 70 %%: СКО с RANSAC от %.3f до %.3f пикс'
      % (er[0], er[-1]))

fig, ax = plt.subplots(figsize=(7.5, 4))
ax.semilogy(fracs * 100, ep, 'o-', label='обычный МНК')
ax.semilogy(fracs * 100, er, 's-', label='RANSAC')
ax.set_xlabel('доля выбросов, %'); ax.set_ylabel('СКО репроекции, пикс')
ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig('/tmp/_ransac.png', dpi=80); plt.close()

Задание 5: при 40 % выбросов — СКО без RANSAC 70.3 пикс, с RANSAC 0.077 пикс
           найдено правильных соответствий 72 из 72, ложных 0


           доля выбросов 0 % -> 70 %: СКО с RANSAC от 0.103 до 0.155 пикс


## Задание 6. Сколько итераций нужно на самом деле

**Ответ на вопрос:** две причины. Первая — формула гарантирует хотя бы одну *полностью чистую* выборку, но частично загрязнённая выборка часто даёт гомографию, достаточно хорошую, чтобы пройти порог. Вторая — финальное переоценивание по всем согласным точкам вытягивает пограничные случаи: даже посредственная гипотеза, собравшая большинство истинных точек, после переоценивания становится точной.

Формулу всё равно следует использовать: она даёт гарантию сверху, а эксперимент — лишь типичное поведение.

In [7]:
def n_iters(eps, s=4, p=0.99):
    return np.log(1 - p) / np.log(1 - (1 - eps) ** s)

print('Задание 6: теоретическое число итераций (s = 4, p = 0,99)')
for e in (0.1, 0.3, 0.5, 0.7):
    emp = []
    for sd in range(40):
        S3, D3, m3 = make_data(frac_out=e, seed=100 + sd)
        cl3 = project_plane(S3[m3], R_true, t_true)
        for it in (1, 2, 3, 5, 8, 12, 20, 35, 60, 100, 200, 400, 900, 2000):
            Hh, _ = ransac_homography(S3, D3, iters=it, seed=sd)
            if np.sqrt(np.mean((apply_H(Hh, S3[m3]) - cl3) ** 2)) < 1.0:
                emp.append(it); break
        else:
            emp.append(2000)
    print('           доля выбросов %2.0f %%: теория %6.0f, эксперимент (медиана) %5.0f'
          % (e * 100, n_iters(e), np.median(emp)))
print('\nВСЕ ПРОВЕРКИ ПРОЙДЕНЫ')

Задание 6: теоретическое число итераций (s = 4, p = 0,99)


           доля выбросов 10 %: теория      4, эксперимент (медиана)     2
           доля выбросов 30 %: теория     17, эксперимент (медиана)     5


           доля выбросов 50 %: теория     71, эксперимент (медиана)    12


           доля выбросов 70 %: теория    566, эксперимент (медиана)   200

ВСЕ ПРОВЕРКИ ПРОЙДЕНЫ


## Критерии приёма работы

| Что проверяется | На что смотреть |
|---|---|
| Задание 1 | DLT реализован сам, а не через `cv2.findHomography`; проверка на контрольных точках есть |
| Задание 2 | Посчитаны оба числа обусловленности; объяснено, почему разница в точности мала |
| Задание 3 | Есть ортогонализация через SVD и обработка знака; куб нарисован поверх маркера |
| Задание 4 | Показана таблица или график убывания невязки по итерациям; названа скорость сходимости |
| Задание 5 | RANSAC реализован полностью, включая финальное переоценивание по согласным точкам |
| Задание 6 | Теория и эксперимент сведены в одну таблицу; расхождение объяснено |
| Выводы | Все вопросы разобраны с числами |
| Воспроизводимость | «Restart & Run All» отрабатывает без ошибок |

**Частые ошибки:**

1. В матрице $A$ перепутан порядок строк или знаки — гомография получается
   транспонированной или зеркальной. Признак: невязка на опорных точках порядка сотен
   пикселей.
2. Берётся первый сингулярный вектор вместо последнего. `np.linalg.svd` возвращает
   $V^{\mathsf T}$, и нужна именно **последняя строка** `Vt[-1]`, а не `Vt[0]`.
3. Забыта нормировка $h_{33} = 1$ — гомография верна, но сравнивать матрицы поэлементно
   невозможно.
4. В `pose_from_homography` не обработан знак: куб уходит за плоскость маркера.
5. В RANSAC отсутствует финальное переоценивание по согласным точкам — результат
   заметно хуже (десятые доли пикселя вместо сотых).
6. Порог согласия задан в метрах вместо пикселей.

## Прикладной блок · PnP и RANSAC в AR-конвейере ROS 2 *(необязательный)*

Пара «задание 4 + задание 5» — это в точности конвейер ROS-трекинга маркеров
(`ros2_aruco`, `fiducials`, Vegard): кадр камеры → найти углы маркера →
`solvePnP` внутри → поза маркера публикуется в `/tf`. Выбросы (блики, ошибочные
уголки) отбрасывает RANSAC — как в задании 5.

| Понятие из работы | Контрагент в ROS 2 |
|---|---|
| `pnp_gauss_newton` | `cv2.solvePnP` внутри `ros2_aruco`: маркер → поза |
| `ransac_homography` + порог 3 пикс | `cv2.findHomography(..., RANSAC, 3.0)` / `solvePnPRansac` |
| `n_iters(eps, s, p)` — оценка итераций | подбор `iterationsCount`/`confidence` в параметрах ROS-нод |
| виртуальный куб поверх маркера (задание 3) | цель всего AR — наложение объекта поверх маркера |

Блок ниже — сверки с OpenCV (расширенные RANSAC-ом). Без OpenCV — пропуск.

In [8]:
# Прикладной блок (необязательный): сверка с cv2.solvePnP / solvePnPRansac
try:
    import cv2
except ImportError:
    cv2 = None

if cv2 is None:
    print('OpenCV не установлен — блок пропущен (сверка необязательная).')
    print('Установка: pip install opencv-python-headless.')
else:
    # 1) cv2.solvePnP против вашего pnp_gauss_newton на чистых наблюдениях
    xi_gn, _ = pnp_gauss_newton(Xw, obs, np.r_[w_true, t_true])
    _, rv, tv = cv2.solvePnP(Xw, obs, K, None, flags=cv2.SOLVEPNP_ITERATIVE)
    print('solvePnP vs ваш Гаусс-Ньютон: dw = %.2e, dt = %.2e'
          % (np.linalg.norm(rv.ravel() - xi_gn[:3]),
             np.linalg.norm(tv.ravel() - xi_gn[3:])))
    assert np.linalg.norm(tv.ravel() - xi_gn[3:]) < 1e-6

    # 2) выбросы в точках: solvePnPRansac против solvePnP без защиты
    obs_bad = obs.copy()
    bad = rng.choice(len(obs_bad), int(0.3 * len(obs_bad)), replace=False)
    obs_bad[bad] += rng.uniform(80, 240, (len(bad), 2))
    _, _, t_r, inl = cv2.solvePnPRansac(
        Xw, obs_bad, K, None, reprojectionError=3.0, iterationsCount=2000)
    _, _, t_plain = cv2.solvePnP(Xw, obs_bad, K, None, flags=cv2.SOLVEPNP_ITERATIVE)
    e_r = np.linalg.norm(t_r.ravel() - t_true)
    e_p = np.linalg.norm(t_plain.ravel() - t_true)
    print('30%% выбросов: ошибка позы с RANSAC = %.4f м, без RANSAC = %.4f м'
          % (e_r, e_p))
    print('inliers: %d из %d (подмешано %d выбросов)'
          % (len(inl), len(obs_bad), len(bad)))
    assert e_r < e_p
    print()
    print('ПРИКЛАДНОЙ БЛОК: ВСЕ ПРОВЕРКИ ПРОЙДЕНЫ')

solvePnP vs ваш Гаусс-Ньютон: dw = 1.11e-16, dt = 1.11e-16
30% выбросов: ошибка позы с RANSAC = 0.0000 м, без RANSAC = 0.9132 м
inliers: 28 из 40 (подмешано 12 выбросов)

ПРИКЛАДНОЙ БЛОК: ВСЕ ПРОВЕРКИ ПРОЙДЕНЫ
